# 01 — Veri Analizi

YKS 2019-2025 konu bazlı soru dağılımı verisi üzerinde keşifsel analiz.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

from config import FEATURES_FILE, TARGET_COL, TRAIN_YEARS, VALID_YEAR, TEST_YEAR

df = pd.read_csv(FEATURES_FILE)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Kolonlar:')
print(df.dtypes)
print()
print('Eksik değerler:')
print(df.isnull().sum())

In [ ]:
print('Yıl dağılımı:')
print(df.groupby('target_year').size())
print()
print('Session dağılımı:')
print(df.groupby('session').size())
print()
print('Field dağılımı:')
print(df.groupby('field').size())

In [ ]:
print('TYT dersleri:')
print(sorted(df[df['session']=='TYT']['subject'].unique()))
print()
print('AYT SAYISAL dersleri:')
print(sorted(df[(df['session']=='AYT') & (df['field']=='SAYISAL')]['subject'].unique()))
print()
print('AYT ESIT_AGIRLIK dersleri:')
print(sorted(df[(df['session']=='AYT') & (df['field']=='ESIT_AGIRLIK')]['subject'].unique()))
print()
print('AYT SOZEL dersleri:')
print(sorted(df[(df['session']=='AYT') & (df['field']=='SOZEL')]['subject'].unique()))

In [ ]:
print('Target dağılımı:')
print(df[TARGET_COL].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df[TARGET_COL].hist(bins=20, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Hedef Soru Sayısı Dağılımı')
axes[0].set_xlabel('Soru Sayısı')

df.groupby('target_year')[TARGET_COL].mean().plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Yıllara Göre Ortalama Soru Sayısı')
axes[1].set_xlabel('Yıl')
plt.tight_layout()
plt.show()

In [ ]:
# Zaman bazlı split boyutları
train = df[df['target_year'].isin(TRAIN_YEARS)]
valid = df[df['target_year'] == VALID_YEAR]
test  = df[df['target_year'] == TEST_YEAR]
print(f'Train (2019-2023): {len(train)} satır')
print(f'Valid (2024):      {len(valid)} satır')
print(f'Test  (2025):      {len(test)} satır')

In [ ]:
# Ders bazlı ortalama soru sayısı
pivot = df.groupby(['target_year', 'subject'])[TARGET_COL].sum().unstack(fill_value=0)
pivot.plot(figsize=(14, 6), marker='o')
plt.title('Yıllara Göre Ders Bazlı Toplam Soru Sayısı')
plt.ylabel('Soru Sayısı')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Korelasyon matrisi (sayısal feature'lar)
num_cols = ['lag_1','lag_2','lag_3','rolling_mean_3_prev','historical_mean_prev',
            'nonzero_frequency_prev','trend_slope_prev', TARGET_COL]
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im)
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45, ha='right')
ax.set_yticklabels(num_cols)
plt.title('Feature Korelasyon Matrisi')
plt.tight_layout()
plt.show()